In [21]:
!pip install torch torchvision torchaudio
!pip install numpy pandas scikit-learn tqdm

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import os
import numpy as np

In [14]:
import os

ROOT = "/content/drive/MyDrive/Hackenza_MUPlovers"
FEATURE_DIR = os.path.join(ROOT, "cache/features_normalized")

print("Feature dir exists:", os.path.exists(FEATURE_DIR))
print("Some files:", os.listdir(FEATURE_DIR)[:10])

Feature dir exists: True
Some files: ['288.npy', '294.npy', '298.npy', '299.npy', '309.npy', '311.npy', '316.npy', '317.npy', '319.npy', '322.npy']


In [16]:
import os
import numpy as np

# Define feature directory (normalized)
ROOT = "/content/drive/MyDrive/Hackenza_MUPlovers"
FEATURE_DIR = os.path.join(ROOT, "cache/features_normalized")

# Get list of .npy files only
file_ids = [
    f.replace(".npy", "")
    for f in os.listdir(FEATURE_DIR)
    if f.endswith(".npy")
]

print("Total feature files:", len(file_ids))

# Load one sample
sample_file = file_ids[0] + ".npy"
sample_path = os.path.join(FEATURE_DIR, sample_file)

sample = np.load(sample_path)

print("File:", sample_file)
print("Shape:", sample.shape)

Total feature files: 160
File: 288.npy
Shape: (24, 783)


In [18]:
import os
import numpy as np

lengths = []

for f in file_ids[:20]:
    arr = np.load(os.path.join(FEATURE_DIR, f + ".npy"))
    lengths.append(arr.shape[0])

print("Min T:", min(lengths))
print("Max T:", max(lengths))
print("Example Ts:", lengths[:10])

Min T: 16
Max T: 79
Example Ts: [24, 23, 22, 33, 16, 71, 23, 59, 23, 39]


In [20]:
import os
import numpy as np

means = []
stds = []

for f in file_ids[:20]:
    arr = np.load(os.path.join(FEATURE_DIR, f + ".npy"))
    means.append(arr.mean())
    stds.append(arr.std())

print("Mean range:", min(means), "to", max(means))
print("Std range:", min(stds), "to", max(stds))

Mean range: -0.007530104 to 0.008054095
Std range: 0.78107995 to 1.397524


In [22]:
import torch
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("CUDA available:", torch.cuda.is_available())
print("Using device:", device)

CUDA available: True
Using device: cuda


In [23]:
from sklearn.metrics import f1_score, accuracy_score

In [24]:
import torch
import torch.nn as nn

class TemporalEncoder(nn.Module):
    def __init__(self, input_dim=783, hidden_dim=256, num_layers=2, dropout=0.3):
        super().__init__()

        self.gru = nn.GRU(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.output_dim = hidden_dim * 2  # because bidirectional

    def forward(self, x):
        # x: [B, T, 783]
        out, _ = self.gru(x)
        # out: [B, T, 2H]
        return out

In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, x, mask=None):
        # x: [B, T, H]

        scores = self.attn(x).squeeze(-1)  # [B, T]

        if mask is not None:
            mask = mask.bool()
            scores = scores.masked_fill(~mask, -1e9)

        weights = F.softmax(scores, dim=1)  # [B, T]

        pooled = torch.sum(x * weights.unsqueeze(-1), dim=1)  # [B, H]

        return pooled, weights

In [26]:
import torch.nn as nn

class ClassifierHead(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout=0.3):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        # x: [B, input_dim]
        logits = self.net(x)
        return logits.squeeze(-1)  # [B]

In [27]:
import torch.nn as nn

class AccentModel(nn.Module):
    def __init__(self, input_dim=783, hidden_dim=256):
        super().__init__()

        self.encoder = TemporalEncoder(input_dim, hidden_dim)
        self.pooling = AttentionPooling(self.encoder.output_dim)
        self.classifier = ClassifierHead(self.encoder.output_dim)

    def forward(self, x, mask=None, return_attention=False):
        # x: [B, T, 783]

        seq_features = self.encoder(x)                 # [B, T, 2H]
        pooled, attn_weights = self.pooling(seq_features, mask)  # [B, 2H]
        logits = self.classifier(pooled)               # [B]

        if return_attention:
            return logits, attn_weights
        else:
            return logits

In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class APLLoss(nn.Module):
    def __init__(self, lambda_apl=0.1):
        super().__init__()
        self.lambda_apl = lambda_apl
        self.eps = 1e-6

    def forward(self, logits, targets):
        # logits: [B]
        # targets: [B] (0 or 1)

        targets = targets.float()

        # Standard BCE (stable version)
        ce = F.binary_cross_entropy_with_logits(logits, targets)

        # Convert logits to probabilities
        probs = torch.sigmoid(logits)
        probs = torch.clamp(probs, self.eps, 1 - self.eps)

        # Reverse Cross Entropy (binary-safe version)
        rce = -torch.mean(
            targets * torch.log(probs) +
            (1 - targets) * torch.log(1 - probs)
        )

        return ce + self.lambda_apl * rce

In [ ]:
B = 4
T = 75
D = 820  # example dimension

x = torch.randn(B, T, D).to(device)
mask = torch.ones(B, T).to(device)
y = torch.randint(0, 2, (B,)).to(device)

model = AccentModel(input_dim=D).to(device)
criterion = APLLoss()

logits, attn = model(x, mask)
loss = criterion(logits, y)

print("Logits shape:", logits.shape)
print("Attention shape:", attn.shape)
print("Loss:", loss.item())

In [29]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset

class AccentDataset(Dataset):
    def __init__(self, file_list, labels_dict, feature_dir):
        """
        file_list: list of file_ids (strings, without .npy)
        labels_dict: dict {file_id: label}
        feature_dir: directory containing .npy feature files
        """
        self.file_list = file_list
        self.labels_dict = labels_dict
        self.feature_dir = feature_dir

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        file_id = str(self.file_list[idx])

        feature_path = os.path.join(self.feature_dir, f"{file_id}.npy")

        if not os.path.exists(feature_path):
            raise FileNotFoundError(f"Missing feature file: {feature_path}")

        x = np.load(feature_path)              # [T, D]
        x = torch.tensor(x, dtype=torch.float32)

        label = torch.tensor(
            self.labels_dict[file_id],
            dtype=torch.float32
        )

        return x, label

In [30]:
from torch.nn.utils.rnn import pad_sequence
import torch

def collate_fn(batch):
    sequences, labels = zip(*batch)

    lengths = [seq.shape[0] for seq in sequences]

    padded_sequences = pad_sequence(sequences, batch_first=True)  # [B, T_max, D]

    # Create mask with same device + dtype consistency
    mask = torch.zeros(
        padded_sequences.size(0),
        padded_sequences.size(1),
        dtype=torch.bool
    )

    for i, length in enumerate(lengths):
        mask[i, :length] = True

    labels = torch.stack(labels)

    return padded_sequences, mask, labels

In [ ]:
from torch.utils.data import DataLoader
import torch.optim as optim
from tqdm import tqdm
from sklearn.metrics import f1_score, accuracy_score

class Trainer:
    def __init__(self, model, train_dataset, val_dataset, lr=5e-4, batch_size=8):
        self.model = model.to(device)
        self.criterion = APLLoss()   # or replace with BCE later
        self.optimizer = optim.Adam(
            self.model.parameters(),
            lr=lr,
            weight_decay=1e-4   # small regularization helps small data
        )

        self.train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            collate_fn=collate_fn
        )

        self.val_loader = DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            collate_fn=collate_fn
        )

    def train_epoch(self):
        self.model.train()
        total_loss = 0

        for x, mask, y in tqdm(self.train_loader):
            x, mask, y = x.to(device), mask.to(device), y.to(device)

            self.optimizer.zero_grad()

            logits = self.model(x, mask)  # ← fixed
            loss = self.criterion(logits, y)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.optimizer.step()

            total_loss += loss.item()

        return total_loss / len(self.train_loader)

    def evaluate(self):
        self.model.eval()

        all_preds = []
        all_labels = []
        all_probs = []

        with torch.no_grad():
            for x, mask, y in self.val_loader:
                x, mask, y = x.to(device), mask.to(device), y.to(device)

                logits = self.model(x, mask)   # ← fixed
                probs = torch.sigmoid(logits)
                preds = (probs > 0.5).float()

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())

        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds)

        return acc, f1, all_probs

In [31]:
def train_model(trainer, epochs=10):
    best_f1 = 0.0

    for epoch in range(epochs):
        train_loss = trainer.train_epoch()
        val_acc, val_f1, val_probs = trainer.evaluate()

        print(f"Epoch {epoch+1}")
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Val Accuracy: {val_acc:.4f}")
        print(f"Val F1 Score: {val_f1:.4f}")

        # Print summary of confidence instead of full list
        print(f"Confidence range: {min(val_probs):.3f} → {max(val_probs):.3f}")
        print("-" * 40)

        # Save best model
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(trainer.model.state_dict(), "best_model.pt")
            print("Saved new best model\n")

    print(f"\nBest Validation F1: {best_f1:.4f}")

In [32]:
import pandas as pd
from torch.utils.data import DataLoader

def run_inference(model, dataset, file_list, batch_size=8):
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn
    )

    model.eval()
    results = []

    idx = 0

    with torch.no_grad():
        for x, mask, _ in loader:  # labels ignored
            x, mask = x.to(device), mask.to(device)

            logits = model(x, mask)   # ← fixed (no unpack)
            probs = torch.sigmoid(logits)

            batch_size_current = probs.size(0)

            for i in range(batch_size_current):
                results.append({
                    "file_id": file_list[idx],
                    "confidence": float(probs[i].cpu().item()),
                    "prediction": int(probs[i] > 0.5)
                })
                idx += 1

    return results

In [34]:
# Recreate model architecture
model = AccentModel(input_dim=783).to(device)

# Load best saved weights
model.load_state_dict(torch.load("best_model.pt"))

print("Loaded best model.")

FileNotFoundError: [Errno 2] No such file or directory: 'best_model.pt'

In [ ]:
import random

num_samples = 10
D = 820

file_ids = []

for i in range(num_samples):
    file_id = f"file_{i:03d}"
    file_ids.append(file_id)

    T = random.randint(60, 90)
    fake_features = np.random.randn(T, D)

    np.save(os.path.join(data_root, f"{file_id}.npy"), fake_features)

print("Created files:", file_ids)

In [ ]:
labels = {fid: random.randint(0, 1) for fid in file_ids}
print(labels)

In [ ]:
train_files = file_ids[:8]
val_files = file_ids[8:]

train_labels = {fid: labels[fid] for fid in train_files}
val_labels = {fid: labels[fid] for fid in val_files}

In [ ]:
train_dataset = AccentDataset(train_files, train_labels, data_root)
val_dataset = AccentDataset(val_files, val_labels, data_root)

In [ ]:
x, y = train_dataset[0]
print("Sample shape:", x.shape)
print("Label:", y)

In [ ]:
model = AccentModel(input_dim=D)

In [ ]:
trainer = Trainer(model, train_dataset, val_dataset, batch_size=4)

In [ ]:
print(trainer.evaluate())

In [ ]:
train_model(trainer, epochs=3)